# Inspect objective weights and progressive barriers

**Question:** How do objective weights and progressive-barrier stopping affect the selected magnetite setting?

Level: 2. Prerequisites: earlier lessons in this topic. Reading time: 30 minutes.



## Objectives

The authored fine-magnetite circuit supplies a one-dimensional grind-target problem. The lesson records endpoint status and exact engine cost for weights 0, 0.5 and 1 under identical bounds. Feasibility, mesh convergence and physical agreement are distinct observed outcomes; none is assumed in advance.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Import the pinned analysis tools once and resolve inputs relative to the notebook folder.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. Inputs are the authored `iron_magnetite_fine` case in the production catalog and `data/derived/contract/operating_contract.json`. No measured plant dataset or economic model is used. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

Three complete production searches hold the magnetite circuit, contract and starts fixed while changing recovered-metal weight to 0, 0.5 and 1. The complementary energy weight is one minus that value. These are preferences in normalized physical outputs, not electricity or commodity prices. Each run retains its own status, endpoint slacks, convergence field and start-level stops. Physical but infeasible endpoints are plotted with explicit markers; refused outcomes remain in the table without invented coordinates. The equal-weight endpoint also receives an independent engine audit, so the interpretation separates preference, physical feasibility and a mesh-based stopping rule.

Set up the magnetite problem to examine objective construction rather than surrogate cost. The nominal recovered mass and specific energy define separate normalization constants. Without those constants a weighted sum would mix tonnes per hour and kWh per tonne arbitrarily. All experiments below share the same bounded grind decision and physical constraints. This controlled setup makes a change in selected operating point attributable to preference weights rather than a changed process definition.

In [ ]:
# Use the authored magnetite circuit as a bounded one-dimensional search example.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.optimization import optimize, Problem
from pipeline.methods.pattern_search import pattern_search
from pipeline.engine.constants import constant

case = CASE_BY_ID["iron_magnetite_fine"]
contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
base = simulate(case.ore, case.plant, case.nominal)
print("Base recovery (%):", round(base.metrics["recovery_pct"], 3))
print("Decision bounds:", contract["cases"][case.id]["inputs"]["target_p80_um"])


First solve the equal-weight problem. Inspect the start-by-start stop reasons and the selected endpoint's grade, power and water slacks. The progressive barrier permits progress toward feasibility when a start is infeasible instead of assigning every such point one indistinguishable penalty. Its objective and violation ordering still require exact process evaluations. A useful terminal point needs both an admissible physical state and a clearly recorded stopping condition.

In [ ]:
# Run the production multistart search without a learned screen, with equal weights on normalized
# recovered metal and energy.
plain = optimize(case, case.nominal, contract, weight=0.5, path=False, screen=None)
print("Unscreened status:", plain["status"])
print("Unscreened evaluations:", plain["evaluations"])
print("Stopping reasons:", [r["stop"] for r in plain["starts"]])
plain_end = plain.get("optimum") or plain.get("least_violating") or plain["base"]
display(pd.Series(plain_end["slacks"], name="Constraint slack").to_frame())


Run the metal-only preference next. This weight does not remove constraints or grant more installed power; it simply removes specific energy from the objective. The resulting endpoint can therefore increase recovered mass only within the same physical feasible region. Keep its energy consumption in the printed comparison, because a solver's definition of better follows its requested objective and does not represent every operating stakeholder's priorities.

In [ ]:
# Change the objective to recovered metal alone while retaining the same bounds, starts and
# physical constraints.
metal_search = optimize(case, case.nominal, contract, weight=1.0, path=False, screen=None)
metal_end = (
    metal_search.get("optimum") or metal_search.get("least_violating") or metal_search["base"]
)
print("Preference:", 1.0)
print("Recovered metal (t/h):", round(metal_end["recovered_tph"], 6))
print("Energy (kWh/t):", round(metal_end["values"]["energy_kwh_t"], 3))


Run the energy-only preference from the same multistart configuration. Energy minimization remains subject to concentrate specification and other physical limits. The outcome is therefore not permission to stop grinding regardless of product quality. Compare its recovered tonnes per hour with the metal-only endpoint to expose what is sacrificed. The three weights are a small computed preference sample, not an exhaustive Pareto frontier or a fitted economic utility function.

In [ ]:
# Change the objective to specific energy alone while retaining the same bounds, starts and
# physical constraints.
energy_search = optimize(case, case.nominal, contract, weight=0.0, path=False, screen=None)
energy_end = (
    energy_search.get("optimum") or energy_search.get("least_violating") or energy_search["base"]
)
print("Preference:", 0.0)
print("Recovered metal (t/h):", round(energy_end["recovered_tph"], 6))
print("Energy (kWh/t):", round(energy_end["values"]["energy_kwh_t"], 3))


The equal-weight preference receives a separate engine audit before comparison with the metal-only and energy-only endpoints. Recovery and energy are printed only for a defined state. This separates a successful circuit calculation from satisfying all optimization constraints and avoids attaching a physical benefit to a refused target.

In [ ]:
# Audit the reported unscreened endpoint only when the engine can define that state.
from pipeline.engine.model import InfeasibleState

plain_point = case.nominal.with_values(**plain_end["decisions"])
plain_check = None
if plain["status"] != "base_refused":
    try:
        plain_check = simulate(case.ore, case.plant, plain_point)
    except InfeasibleState as refusal:
        print("Unscreened audit refusal:", refusal.code)
print("Unscreened audit available:", plain_check is not None)
if plain_check is not None:
    print("Unscreened engine recovery (%):", round(plain_check.metrics["recovery_pct"], 3))
    print(
        "Unscreened engine energy (kWh/t):",
        round(plain_check.metrics["specific_energy_total_kwh_t"], 3),
    )


Inspect the equal-weight start's objective and mesh histories. Accepted improvements can expand or maintain mesh, while unsuccessful iterations contract it. The two panels distinguish physical-evaluation work from iteration count. Because multistarts can reuse the problem's cache, summing per-start histories should not be substituted for the total unique engine-call count reported by the optimizer. Read each count according to the quantity it actually records.

In [ ]:
# Plot the recorded incumbent objective and mesh history of the winning unscreened start.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 8})
history = pd.DataFrame(
    plain.get("trace", []), columns=["Outcome", "Mesh", "Barrier", "Objective", "Evaluations"]
)
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
axes[0].plot(history["Evaluations"], history["Objective"], marker=".")
axes[0].set(xlabel="Exact calls (n)", ylabel="Objective (1)", title="Objective progress")
axes[1].step(range(len(history)), history["Mesh"], where="post")
axes[1].set(xlabel="Iteration (n)", ylabel="Mesh (1)", title="Search mesh")
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


The trace supports a narrow convergence statement: the selected start stopped under the printed rule after its recorded sequence of evaluations. It cannot demonstrate global optimality on every possible target, nor establish robustness against a different ore or uncertain work index. A flat objective near the end may reflect the declared improvement tolerance rather than a physically exact plateau. Preserve the stop reason when quoting the terminal recovery and energy.

Plot the three computed endpoints in energy and recovered-metal coordinates. A weighted score hides this relationship, whereas process units let a reader see the trade-off directly. Connecting the three markers only guides reading order. It does not fill in uncomputed weights, guarantee convexity or establish that every point on the line is feasible. The endpoint slacks remain the evidence for each computed candidate's admissibility.

In [ ]:
# Preserve each run status in the physical preference plot.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 80})
preference_runs = (("Equal", plain), ("Metal", metal_search), ("Energy", energy_search))
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for label, run in preference_runs:
    end = run.get("optimum") or run.get("least_violating") or run["base"]
    if run["status"] != "base_refused" and end["values"]:
        marker = "o" if end["feasible"] else "x"
        ax.scatter(
            end["values"]["energy_kwh_t"],
            end["recovered_tph"],
            marker=marker,
            label=label + " [" + run["status"] + "]",
        )
ax.set(
    xlabel="Specific energy (kWh/t)",
    ylabel="Recovered metal\n(t/h)",
    title="Computed preference endpoints",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)


Each marker is one computed preference endpoint and its legend retains the run status. Crosses identify physically solvable but infeasible results; refused states supply no invented point. The status table below preserves all runs, including unavailable outputs. No connecting line implies an interpolated feasible frontier, and the normalized weights do not assign economic prices to electricity or recovered metal.

Constraint violation is formed from normalized negative slacks beyond the declared feasibility tolerance. Squaring and summing these violations gives the barrier quantity used by pattern search. It is distinct from the energy/metal objective: a candidate can improve feasibility while worsening the weighted objective. This distinction explains why a trace may spend evaluations without raising its feasible incumbent. The multistart cache records unique physical decision vectors shared across starts, while the displayed winning trace counts work within one search. Do not infer the total by multiplying its length by the number of starts.

All three preference runs remain in a common status table. The metal-only, equal-weight and energy-only endpoints retain their own slacks and stops, so a more attractive mass/energy coordinate cannot conceal infeasibility. Missing physical output from a refused problem remains absent while its run status and decision input are still visible.

In [ ]:
# Report each run before comparing cost or preference.
runs = {"Equal weight": plain, "Metal only": metal_search, "Energy only": energy_search}
endpoint_rows = []
for label, run in runs.items():
    end = run.get("optimum") or run.get("least_violating") or run["base"]
    physical = run["status"] != "base_refused" and bool(end["values"])
    endpoint_rows.append(
        {
            "Run": label,
            "Status": run["status"],
            "Feasible": end["feasible"],
            "Converged": run.get("converged"),
            "Stops": ", ".join((r["stop"] for r in run["starts"])),
            "Decisions": end["decisions"],
            "Slacks": end["slacks"],
            "Exact calls": run["evaluations"],
            "Recovered metal (t/h)": end["recovered_tph"] if physical else None,
            "Recovery (%)": end["recovery_pct"] if physical else None,
            "Specific energy (kWh/t)": end["values"].get("energy_kwh_t") if physical else None,
        }
    )
endpoints = pd.DataFrame(endpoint_rows)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(endpoints)


## Results

**Answer:** The equal-weight target is <!-- result:Reported target P80 (µm) -->0<!-- /result --> µm with feasibility flag <!-- result:Selected feasible (1=yes) -->0<!-- /result -->. Metal-only and energy-only flags are <!-- result:Metal endpoint feasible (1=yes) -->0<!-- /result --> and <!-- result:Energy endpoint feasible (1=yes) -->0<!-- /result --> (1=yes). Their separately labelled physical outcomes and slacks show the preference trade-off, while the per-start stops and reported convergence qualify termination. Unavailable physical outcomes remain absent rather than becoming plotted zero values.

Summarize the equal-weight target and feasibility flags of all three preference endpoints after inspecting their status table.

In [ ]:
# Collect status-qualified search outcomes without inventing missing physical metrics.
answer_values = {
    "Unscreened count": plain["evaluations"],
    "Reported target P80 (µm)": plain_end["decisions"]["target_p80_um"],
    "Selected feasible (1=yes)": int(plain_end["feasible"]),
    "Convergence reported (1=yes)": int(plain.get("converged") is True),
}
answer_values.update(
    {
        "Metal endpoint feasible (1=yes)": int(metal_end["feasible"]),
        "Energy endpoint feasible (1=yes)": int(energy_end["feasible"]),
    }
)
answer_values = {label: round(float(number), 8) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

The three preference endpoints retain individual feasibility and stopping information. Differences in recovered mass and energy follow from the specified normalized weights, without an economic utility claim or an interpolated feasible frontier.

## References

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662